In [ ]:
%pip install pymc arviz

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import statsmodels.api as sm

ROOT = Path.cwd().resolve()
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

rets = pd.read_csv(ROOT / "data" / "processed" / "return_panel_500.csv", parse_dates=["date"])
rets["ticker"] = rets["ticker"].astype(str)
rets = rets.sort_values(["ticker", "date"]).reset_index(drop=True)
rets["ret_fwd"] = rets.groupby("ticker")["return"].shift(-1)

a = pd.read_csv(OUT / "portfolio_A_core_358.csv")
b = pd.read_csv(OUT / "portfolio_B_price_only_500.csv")
c = pd.read_csv(OUT / "portfolio_C_overlap.csv")

def prep_model_a():
    fv = pd.read_csv(OUT / "factor_A_value_358.csv")
    fq = pd.read_csv(OUT / "factor_A_quality_358.csv")
    fs = pd.read_csv(OUT / "factor_A_sue_358.csv")
    fm = pd.read_csv(OUT / "factor_A_momentum_358.csv", parse_dates=["date"])

    fv["report_date"] = pd.to_datetime(fv["report_date"])
    fq["report_date"] = pd.to_datetime(fq["report_date"])
    fs["report_date"] = pd.to_datetime(fs["report_date"])
    fm = fm.rename(columns={"date": "report_date"})
    fm["report_date"] = pd.to_datetime(fm["report_date"])

    x = fv.merge(fq[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
    x = x.merge(fs[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
    x = x.merge(fm[["ticker", "report_date", "momentum_12m_z"]], on=["ticker", "report_date"], how="inner")
    x["report_date"] = pd.to_datetime(x["report_date"])
    x = x.merge(rets[["date", "ticker", "ret_fwd"]], left_on=["report_date", "ticker"], right_on=["date", "ticker"], how="inner")
    x = x.dropna(subset=["ret_fwd", "value_score", "quality_score", "sue_z", "momentum_12m_z"]).copy()
    return x

def prep_model_b():
    x = pd.read_csv(OUT / "factor_B_price_only_500.csv", parse_dates=["date"])
    x = x.merge(rets[["date", "ticker", "ret_fwd"]], on=["date", "ticker"], how="inner")
    x = x.dropna(subset=["ret_fwd", "momentum_12m_z"]).copy()
    return x

def prep_model_c():
    fv = pd.read_csv(OUT / "factor_C_value_overlap.csv")
    fq = pd.read_csv(OUT / "factor_C_quality_overlap.csv")
    fs = pd.read_csv(OUT / "factor_C_sue_overlap.csv")
    fm = pd.read_csv(OUT / "factor_C_momentum_overlap.csv", parse_dates=["date"])

    fv["report_date"] = pd.to_datetime(fv["report_date"])
    fq["report_date"] = pd.to_datetime(fq["report_date"])
    fs["report_date"] = pd.to_datetime(fs["report_date"])
    fm = fm.rename(columns={"date": "report_date"})
    fm["report_date"] = pd.to_datetime(fm["report_date"])

    x = fv.merge(fq[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
    x = x.merge(fs[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
    x = x.merge(fm[["ticker", "report_date", "momentum_12m_z"]], on=["ticker", "report_date"], how="inner")
    x["report_date"] = pd.to_datetime(x["report_date"])
    x = x.merge(rets[["date", "ticker", "ret_fwd"]], left_on=["report_date", "ticker"], right_on=["date", "ticker"], how="inner")
    x = x.dropna(subset=["ret_fwd", "value_score", "quality_score", "sue_z", "momentum_12m_z"]).copy()
    return x

def fit_ols(df, y, xcols, name):
    X = df[xcols].copy()
    X = (X - X.mean()) / X.std()
    X = sm.add_constant(X)
    yv = (df[y] - df[y].mean()) / df[y].std()

    model = sm.OLS(yv, X).fit()

    summ = model.summary2().tables[1].reset_index().rename(columns={"index": "term"})
    summ["model"] = name
    summ.to_csv(OUT / f"ols_{name}_summary.csv", index=False)
    return model, summ

a_df = prep_model_a()
b_df = prep_model_b()
c_df = prep_model_c()
print(len(a_df), len(b_df), len(c_df))

print("Starting model A...")
model_a, summ_a = fit_ols(a_df, "ret_fwd", ["value_score", "quality_score", "sue_z", "momentum_12m_z"], "A_core_358")
print("Model A done. Starting model B...")
model_b, summ_b = fit_ols(b_df, "ret_fwd", ["momentum_12m_z"], "B_price_only_500")
print("Model B done. Starting model C...")
model_c, summ_c = fit_ols(c_df, "ret_fwd", ["value_score", "quality_score", "sue_z", "momentum_12m_z"], "C_overlap")
print("Model C done.")

pd.concat([summ_a, summ_b, summ_c], ignore_index=True).to_csv(OUT / "ols_all_summaries.csv", index=False)

coverage = pd.DataFrame([
    {"model": "A_core_358", "n_obs": len(a_df), "n_tickers": a_df["ticker"].nunique()},
    {"model": "B_price_only_500", "n_obs": len(b_df), "n_tickers": b_df["ticker"].nunique()},
    {"model": "C_overlap", "n_obs": len(c_df), "n_tickers": c_df["ticker"].nunique()},
])
coverage.to_csv(OUT / "bayes_coverage.csv", index=False)

print(coverage)
print(summ_a)
print(summ_b)
print(summ_c)

4822 64650 4822
Starting model A...
Model A done. Starting model B...
Model B done. Starting model C...
Model C done.
              model  n_obs  n_tickers
0        A_core_358   4822        325
1  B_price_only_500  64650        498
2         C_overlap   4822        325
             term         Coef.  Std.Err.             t     P>|t|    [0.025  \
0           const  2.114194e-18  0.014388  1.469420e-16  1.000000 -0.028207   
1     value_score -3.627765e-02  0.015704 -2.310135e+00  0.020923 -0.067064   
2   quality_score  2.987182e-02  0.015651  1.908608e+00  0.056372 -0.000811   
3           sue_z -3.517602e-03  0.014452 -2.433987e-01  0.807707 -0.031850   
4  momentum_12m_z  3.142509e-02  0.014514  2.165169e+00  0.030424  0.002971   

     0.975]       model  
0  0.028207  A_core_358  
1 -0.005491  A_core_358  
2  0.060555  A_core_358  
3  0.024815  A_core_358  
4  0.059879  A_core_358  
             term         Coef.  Std.Err.             t         P>|t|  \
0           const -7.23874